In [2]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 15.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 3.8 MB/s eta 0:00:00


In [3]:

from pathlib import Path
import json
import math
import time

import kagglehub
import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError


# Configuration
DATASET_ID = "sayedgamal99/smoke-fire-detection-yolo"
EXPECTED_SPLITS = {
    "train": 14122,
    "val": 3099,
    "test": 4306,
}
CLASS_NAMES = {0: "smoke", 1: "fire"}

OUTPUT_DIR = Path("/kaggle/working/dfire_data_pipeline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

start_time = time.time()


# Download dataset
download_path = Path(kagglehub.dataset_download(DATASET_ID))
print(f"Downloaded dataset to: {download_path}")


# Locate the dataset configuration
yaml_files = list(download_path.rglob("data.yaml"))

if not yaml_files:
    raise FileNotFoundError(
        f"Could not find data.yaml inside {download_path}"
    )

yaml_path = next(
    (
        path for path in yaml_files
        if all(
            (path.parent / "data" / split / "images").is_dir()
            for split in EXPECTED_SPLITS
        )
    ),
    None,
)

if yaml_path is None:
    raise FileNotFoundError(
        "Could not locate a data.yaml next to "
        "data/train/images, data/val/images and data/test/images."
    )

dataset_root = yaml_path.parent
print(f"Dataset root: {dataset_root}")
print(f"Configuration: {yaml_path}")


# Verify class configuration
import yaml

with yaml_path.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

names = config.get("names")
if isinstance(names, list):
    names = {i: name for i, name in enumerate(names)}
elif isinstance(names, dict):
    names = {int(k): v for k, v in names.items()}

if names != CLASS_NAMES:
    raise ValueError(
        f"Unexpected classes: {names}. "
        f"Expected {CLASS_NAMES}."
    )

print(f"Classes verified: {names}")


# Audit images and YOLO annotations
summary = {
    "dataset_id": DATASET_ID,
    "dataset_root": str(dataset_root),
    "configuration": str(yaml_path),
    "classes": {str(k): v for k, v in CLASS_NAMES.items()},
    "splits": {},
    "total_images": 0,
    "total_valid_boxes": 0,
    "class_box_counts": {"smoke": 0, "fire": 0},
    "total_empty_labels": 0,
    "total_missing_labels": 0,
    "total_orphan_labels": 0,
    "total_invalid_rows": 0,
    "total_out_of_bounds_boxes": 0,
    "total_unreadable_images": 0,
}

issues = []
negative_candidates = []

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for split, expected_count in EXPECTED_SPLITS.items():
    split_root = dataset_root / "data" / split
    images_dir = split_root / "images"
    labels_dir = split_root / "labels"

    if not images_dir.is_dir() or not labels_dir.is_dir():
        raise FileNotFoundError(
            f"Missing images or labels directory for {split}: "
            f"{split_root}"
        )

    images = sorted(
        p for p in images_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in image_extensions
    )
    label_files = {
        p.relative_to(labels_dir).with_suffix("")
        for p in labels_dir.rglob("*.txt")
        if p.is_file()
    }

    image_stems = {
        p.relative_to(images_dir).with_suffix("")
        for p in images
    }

    missing_labels = image_stems - label_files
    orphan_labels = label_files - image_stems

    split_stats = {
        "expected_images": expected_count,
        "actual_images": len(images),
        "image_count_matches": len(images) == expected_count,
        "label_files": len(label_files),
        "missing_labels": len(missing_labels),
        "orphan_labels": len(orphan_labels),
        "empty_labels": 0,
        "valid_boxes": 0,
        "invalid_rows": 0,
        "out_of_bounds_boxes": 0,
        "unreadable_images": 0,
        "class_box_counts": {"smoke": 0, "fire": 0},
    }

    for relative in sorted(missing_labels):
        issues.append({
            "split": split,
            "image": str(relative),
            "issue": "missing_label_file",
            "details": "",
        })

    for relative in sorted(orphan_labels):
        issues.append({
            "split": split,
            "image": str(relative),
            "issue": "orphan_label_file",
            "details": "",
        })

    for image_path in images:
        relative = image_path.relative_to(images_dir)
        label_path = labels_dir / relative.with_suffix(".txt")
        relative_str = str(relative)

        try:
            with Image.open(image_path) as image:
                image.verify()
        except (OSError, UnidentifiedImageError, ValueError) as exc:
            split_stats["unreadable_images"] += 1
            issues.append({
                "split": split,
                "image": relative_str,
                "issue": "unreadable_image",
                "details": str(exc),
            })

        if not label_path.exists():
            continue

        text = label_path.read_text(encoding="utf-8").strip()

        if not text:
            split_stats["empty_labels"] += 1
            negative_candidates.append({
                "split": split,
                "image": str(image_path),
                "label": str(label_path),
                "review_status": "candidate_unverified",
            })
            continue

        for line_number, line in enumerate(text.splitlines(), start=1):
            parts = line.split()

            try:
                if len(parts) != 5:
                    raise ValueError("Expected 5 values")

                cls_float, x, y, w, h = map(float, parts)

                if not all(
                    math.isfinite(v)
                    for v in (cls_float, x, y, w, h)
                ):
                    raise ValueError("Non-finite value")

                if cls_float not in (0.0, 1.0):
                    raise ValueError("Unknown class ID")

                if w <= 0 or h <= 0:
                    raise ValueError("Non-positive box size")

                if not (0 <= x <= 1 and 0 <= y <= 1):
                    raise ValueError("Invalid normalized center")

                if w > 1 or h > 1:
                    raise ValueError("Invalid normalized box size")

                cls = int(cls_float)

            except ValueError as exc:
                split_stats["invalid_rows"] += 1
                issues.append({
                    "split": split,
                    "image": relative_str,
                    "issue": "invalid_label_row",
                    "details": f"line={line_number}; {exc}; {line}",
                })
                continue

            split_stats["valid_boxes"] += 1
            class_name = CLASS_NAMES[cls]
            split_stats["class_box_counts"][class_name] += 1

            x1, y1 = x - w / 2, y - h / 2
            x2, y2 = x + w / 2, y + h / 2

            if x1 < 0 or y1 < 0 or x2 > 1 or y2 > 1:
                split_stats["out_of_bounds_boxes"] += 1
                issues.append({
                    "split": split,
                    "image": relative_str,
                    "issue": "box_out_of_bounds",
                    "details": (
                        f"line={line_number}; class={class_name}; "
                        f"x={x}; y={y}; w={w}; h={h}"
                    ),
                })

    summary["splits"][split] = split_stats
    summary["total_images"] += len(images)
    summary["total_valid_boxes"] += split_stats["valid_boxes"]
    summary["total_empty_labels"] += split_stats["empty_labels"]
    summary["total_missing_labels"] += split_stats["missing_labels"]
    summary["total_orphan_labels"] += split_stats["orphan_labels"]
    summary["total_invalid_rows"] += split_stats["invalid_rows"]
    summary["total_out_of_bounds_boxes"] += (
        split_stats["out_of_bounds_boxes"]
    )
    summary["total_unreadable_images"] += (
        split_stats["unreadable_images"]
    )

    for name, count in split_stats["class_box_counts"].items():
        summary["class_box_counts"][name] += count

    print(
        f"{split}: images={len(images)} "
        f"(expected {expected_count}), "
        f"boxes={split_stats['valid_boxes']}, "
        f"empty_labels={split_stats['empty_labels']}, "
        f"invalid_rows={split_stats['invalid_rows']}, "
        f"out_of_bounds={split_stats['out_of_bounds_boxes']}"
    )


# Save the audit results
summary["audit_seconds"] = round(time.time() - start_time, 2)
summary["audit_passed"] = (
    all(
        stats["image_count_matches"]
        for stats in summary["splits"].values()
    )
    and summary["total_missing_labels"] == 0
    and summary["total_orphan_labels"] == 0
    and summary["total_invalid_rows"] == 0
    and summary["total_out_of_bounds_boxes"] == 0
    and summary["total_unreadable_images"] == 0
)

summary_path = OUTPUT_DIR / "dataset_audit_summary.json"
issues_path = OUTPUT_DIR / "dataset_audit_issues.csv"
negatives_path = OUTPUT_DIR / "hard_negative_candidates.csv"

summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

pd.DataFrame(
    issues,
    columns=["split", "image", "issue", "details"],
).to_csv(issues_path, index=False)

pd.DataFrame(
    negative_candidates,
    columns=["split", "image", "label", "review_status"],
).to_csv(negatives_path, index=False)

print("\n=== FINAL AUDIT SUMMARY ===")
print(json.dumps(summary, indent=2, ensure_ascii=False))
print("\n=== OUTPUT FILES ===")
print(summary_path)
print(issues_path)
print(negatives_path)


Downloaded dataset to: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo
Dataset root: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo
Configuration: /kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data.yaml
Classes verified: {0: 'smoke', 1: 'fire'}


KeyboardInterrupt: 

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

OUTPUT_DIR = Path("/kaggle/working/dfire_data_pipeline")
issues_path = OUTPUT_DIR / "dataset_audit_issues.csv"

issues = pd.read_csv(issues_path)

print("=== ISSUE COUNTS ===")
print(
    issues.groupby(["split", "issue"])
    .size()
    .unstack(fill_value=0)
    .to_string()
)

invalid = issues[issues["issue"] == "invalid_label_row"]

print("\n=== INVALID LABEL ROWS ===")
if invalid.empty:
    print("No invalid rows.")
else:
    print(invalid[["split", "image", "details"]].to_string(index=False))

bounds = issues[issues["issue"] == "box_out_of_bounds"].copy()

def inspect_box(details):
    values = {}
    for item in str(details).split(";"):
        if "=" in item:
            key, value = item.strip().split("=", 1)
            values[key.strip()] = value.strip()

    try:
        x = float(values["x"])
        y = float(values["y"])
        w = float(values["w"])
        h = float(values["h"])

        x1, y1 = x - w / 2, y - h / 2
        x2, y2 = x + w / 2, y + h / 2

        overflow = max(0.0, -x1, -y1, x2 - 1.0, y2 - 1.0)

        if overflow <= 1e-3:
            severity = "small"
        elif overflow <= 0.01:
            severity = "moderate"
        else:
            severity = "large"

        return pd.Series({
            "class": values.get("class", "unknown"),
            "overflow": overflow,
            "severity": severity,
        })
    except (KeyError, ValueError):
        return pd.Series({
            "class": values.get("class", "unknown"),
            "overflow": np.nan,
            "severity": "unparsed",
        })

if not bounds.empty:
    parsed = bounds["details"].apply(inspect_box)
    bounds = pd.concat(
        [bounds.reset_index(drop=True), parsed.reset_index(drop=True)],
        axis=1,
    )

    print("\n=== BOX OVERFLOW SEVERITY ===")
    print(
        bounds.groupby(["split", "severity"])
        .size()
        .unstack(fill_value=0)
        .to_string()
    )

    print("\n=== LARGEST OVERFLOWS ===")
    print(
        bounds.nlargest(10, "overflow")[
            ["split", "image", "details", "overflow", "severity"]
        ].to_string(index=False)
    )

    review_path = OUTPUT_DIR / "bounding_box_review.csv"
    bounds.to_csv(review_path, index=False)
    print(f"\nDetailed review saved to: {review_path}")

print("\nOriginal dataset has not been modified.")


In [ ]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

ROOT = Path("/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo")
OUT = Path("/kaggle/working/dfire_data_pipeline")

review = pd.read_csv(OUT / "bounding_box_review.csv")
review = review.sort_values("overflow", ascending=False).head(12)

class_names = {0: "smoke", 1: "fire"}
colors = {"smoke": "orange", "fire": "red"}
review_results = []

fig, axes = plt.subplots(3, 4, figsize=(20, 14))
axes = axes.flatten()

for ax, (_, row) in zip(axes, review.iterrows()):
    split = row["split"]
    filename = row["image"]

    image_path = ROOT / "data" / split / "images" / filename
    if not image_path.exists():
        matches = list(
            (ROOT / "data" / split / "images").rglob(
                Path(filename).name
            )
        )
        if not matches:
            ax.set_title(f"Image not found: {filename}")
            ax.axis("off")
            continue
        image_path = matches[0]

    with Image.open(image_path) as im:
        image = im.convert("RGB")
        width, height = image.size

    ax.imshow(image)

    label_path = (
        ROOT / "data" / split / "labels" /
        Path(filename).with_suffix(".txt")
    )

    drawn_boxes = 0
    invalid_boxes = 0

    if label_path.exists():
        for line in label_path.read_text().splitlines():
            parts = line.split()

            try:
                if len(parts) != 5:
                    invalid_boxes += 1
                    continue

                cls, x, y, w, h = map(float, parts)

                if (
                    cls not in class_names
                    or w <= 0 or h <= 0
                    or not all(
                        pd.notna(v) and abs(v) != float("inf")
                        for v in [cls, x, y, w, h]
                    )
                ):
                    invalid_boxes += 1
                    continue

                x1 = (x - w / 2) * width
                y1 = (y - h / 2) * height
                box_w = w * width
                box_h = h * height
                name = class_names[int(cls)]

                ax.add_patch(Rectangle(
                    (x1, y1),
                    box_w,
                    box_h,
                    linewidth=2,
                    edgecolor=colors[name],
                    facecolor="none",
                    clip_on=True,
                ))
                drawn_boxes += 1

            except (ValueError, OverflowError):
                invalid_boxes += 1

    ax.set_title(
        f"{split}: {Path(filename).name}\n"
        f"Overflow={row['overflow']:.3f}; "
        f"boxes={drawn_boxes}; invalid={invalid_boxes}",
        fontsize=9,
    )
    ax.axis("off")

    review_results.append({
        "split": split,
        "image": str(image_path),
        "overflow": row["overflow"],
        "manual_status": "",
        "notes": "",
    })

plt.tight_layout()
plt.show()

review_path = OUT / "manual_visual_review.csv"
pd.DataFrame(review_results).to_csv(review_path, index=False)

print(f"Visual review list saved to: {review_path}")
print("Orange boxes = smoke; red boxes = fire.")
print("Boxes are drawn using the original annotation coordinates.")


In [ ]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
import math

ROOT = Path("/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo")
OUT = Path("/kaggle/working/dfire_data_pipeline")

issues = pd.read_csv(OUT / "dataset_audit_issues.csv")
invalid = issues[issues["issue"] == "invalid_label_row"].reset_index(drop=True)

class_names = {0: "smoke", 1: "fire"}
colors = {"smoke": "orange", "fire": "red"}

for start in range(0, len(invalid), 8):
    batch = invalid.iloc[start:start + 8]
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()

    for ax, (_, row) in zip(axes, batch.iterrows()):
        split = row["split"]
        filename = Path(row["image"]).name
        image_path = ROOT / "data" / split / "images" / filename
        label_path = ROOT / "data" / split / "labels" / Path(filename).with_suffix(".txt")

        with Image.open(image_path) as im:
            image = im.convert("RGB")
            width, height = image.size

        ax.imshow(image)
        lines = label_path.read_text(encoding="utf-8").splitlines()
        detail = str(row["details"])
        line_number = int(detail.split("line=")[1].split(";")[0])

        for i, line in enumerate(lines, start=1):
            parts = line.split()
            if len(parts) != 5:
                continue

            try:
                cls, x, y, w, h = map(float, parts)
                if not all(math.isfinite(v) for v in [cls, x, y, w, h]):
                    continue
                if cls not in class_names or w <= 0 or h <= 0:
                    continue

                x1 = (x - w / 2) * width
                y1 = (y - h / 2) * height
                box_w = w * width
                box_h = h * height
                name = class_names[int(cls)]

                ax.add_patch(Rectangle(
                    (x1, y1), box_w, box_h,
                    linewidth=3 if i == line_number else 1.5,
                    edgecolor="magenta" if i == line_number else colors[name],
                    facecolor="none",
                    clip_on=True,
                ))
            except (ValueError, OverflowError):
                continue

        ax.set_title(
            f"{split}: {filename}\nProblem at line {line_number}",
            fontsize=9,
        )
        ax.axis("off")

    for ax in axes[len(batch):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

print("Magenta = problematic annotation row.")
print("Orange = smoke; red = fire.")
print("Images and labels have not been modified.")


In [ ]:

from pathlib import Path
import csv
import json
import math

SOURCE = Path("/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo")
OUTPUT = Path("/kaggle/working/dfire_prepared")
DATA = OUTPUT / "data"

SPLITS = {
    "train": 14122,
    "val": 3099,
    "test": 4306,
}
CLASS_NAMES = {0: "smoke", 1: "fire"}

OUTPUT.mkdir(parents=True, exist_ok=True)

# 1. Prepare directories and make sure images are available
for split, expected in SPLITS.items():
    source_split = SOURCE / "data" / split
    source_images = source_split / "images"
    source_labels = source_split / "labels"

    if not source_images.is_dir() or not source_labels.is_dir():
        raise FileNotFoundError(
            f"Source images or labels are missing: {source_split}"
        )

    prepared_split = DATA / split
    prepared_images = prepared_split / "images"
    prepared_labels = prepared_split / "labels"

    prepared_split.mkdir(parents=True, exist_ok=True)

    if not prepared_images.exists():
        prepared_images.symlink_to(
            source_images, target_is_directory=True
        )
    elif not any(prepared_images.iterdir()):
        if prepared_images.is_symlink():
            prepared_images.unlink()
        else:
            prepared_images.rmdir()
        prepared_images.symlink_to(
            source_images, target_is_directory=True
        )

    prepared_labels.mkdir(parents=True, exist_ok=True)

    image_count = sum(
        p.is_file() and p.suffix.lower() in
        {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        for p in prepared_images.iterdir()
    )

    if image_count != expected:
        raise ValueError(
            f"{split}: expected {expected} images, found {image_count}"
        )

print("Image directories checked.")

# 2. Clean annotations into the prepared labels directories
summary = {}
issues = []
total_class_counts = {"smoke": 0, "fire": 0}

for split, expected in SPLITS.items():
    original_labels = SOURCE / "data" / split / "labels"
    prepared_labels = DATA / split / "labels"

    label_files = sorted(original_labels.glob("*.txt"))

    if len(label_files) != expected:
        raise ValueError(
            f"{split}: expected {expected} label files, "
            f"found {len(label_files)}"
        )

    counts = {"smoke": 0, "fire": 0}
    empty = removed = clipped = 0

    for source_label in label_files:
        clean_lines = []

        for line_no, line in enumerate(
            source_label.read_text(encoding="utf-8-sig").splitlines(),
            start=1,
        ):
            if not line.strip():
                continue

            try:
                parts = line.split()
                if len(parts) != 5:
                    raise ValueError("Expected 5 values")

                cls_value = float(parts[0])
                x, y, w, h = map(float, parts[1:])
                values = [cls_value, x, y, w, h]

                if not all(math.isfinite(v) for v in values):
                    raise ValueError("Non-finite value")
                if not cls_value.is_integer():
                    raise ValueError("Non-integer class ID")

                cls = int(cls_value)
                if cls not in CLASS_NAMES:
                    raise ValueError("Unknown class ID")
                if not (0 <= x <= 1 and 0 <= y <= 1):
                    raise ValueError("Center outside image")
                if not (0 < w <= 1 and 0 < h <= 1):
                    raise ValueError("Invalid box size")

                # Clip boxes to image boundaries.
                x1, y1 = x - w / 2, y - h / 2
                x2, y2 = x + w / 2, y + h / 2

                a, b = max(0.0, x1), max(0.0, y1)
                c, d = min(1.0, x2), min(1.0, y2)

                if c <= a or d <= b:
                    raise ValueError("Box has no visible area")

                if (a, b, c, d) != (x1, y1, x2, y2):
                    clipped += 1

                nw, nh = c - a, d - b
                nx, ny = (a + c) / 2, (b + d) / 2

                clean_lines.append(
                    f"{cls} {nx:.8f} {ny:.8f} "
                    f"{nw:.8f} {nh:.8f}"
                )

                counts[CLASS_NAMES[cls]] += 1
                total_class_counts[CLASS_NAMES[cls]] += 1

            except (ValueError, OverflowError) as error:
                removed += 1
                issues.append({
                    "split": split,
                    "image": source_label.stem,
                    "line": line_no,
                    "reason": str(error),
                    "original_line": line,
                })

        (prepared_labels / source_label.name).write_text(
            "\n".join(clean_lines) + ("\n" if clean_lines else ""),
            encoding="utf-8",
        )

        if not clean_lines:
            empty += 1

    summary[split] = {
        "images": expected,
        "labels_written": len(label_files),
        "empty_labels": empty,
        "smoke_boxes": counts["smoke"],
        "fire_boxes": counts["fire"],
        "total_boxes": sum(counts.values()),
        "removed_invalid_rows": removed,
        "clipped_boxes": clipped,
    }

# 3. Save reports
with (OUTPUT / "removed_label_rows.csv").open(
    "w", newline="", encoding="utf-8"
) as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "split", "image", "line", "reason", "original_line"
        ],
    )
    writer.writeheader()
    writer.writerows(issues)

report = {
    "splits": summary,
    "total_images": sum(SPLITS.values()),
    "total_boxes": sum(total_class_counts.values()),
    "class_counts": total_class_counts,
    "removed_invalid_rows": len(issues),
    "policy": (
        "Invalid annotation rows rejected; boxes crossing image "
        "boundaries clipped to visible image bounds."
    ),
}

(OUTPUT / "dataset_summary.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\n=== PREPARATION COMPLETE ===")
for split, stats in summary.items():
    print(f"\n{split.upper()}:")
    for key, value in stats.items():
        print(f"  {key}: {value}")

print("\nTotal boxes:", report["total_boxes"])
print("Class counts:", total_class_counts)
print("Invalid rows removed:", len(issues))
print("Report:", OUTPUT / "dataset_summary.json")
print("Original dataset unchanged.")


In [ ]:

from pathlib import Path
from PIL import Image
import math

DATA = Path("/kaggle/working/dfire_prepared/data")
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for split in ("train", "val", "test"):
    label_dir = DATA / split / "labels"
    image_dir = DATA / split / "images"

    image_by_stem = {
        p.stem: p
        for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    }

    checked = False

    for label_path in sorted(label_dir.glob("*.txt")):
        lines = [
            line.split()
            for line in label_path.read_text(
                encoding="utf-8"
            ).splitlines()
            if line.strip()
        ]

        if not lines:
            continue

        image_path = image_by_stem.get(label_path.stem)
        assert image_path is not None, (
            f"Image not found for {split}/{label_path.name}"
        )

        with Image.open(image_path) as im:
            image_size = im.size
            im.verify()

        for parts in lines:
            assert len(parts) == 5
            cls, x, y, w, h = map(float, parts)

            assert cls in (0, 1)
            assert all(
                math.isfinite(v) for v in (x, y, w, h)
            )
            assert 0 <= x <= 1 and 0 <= y <= 1
            assert 0 < w <= 1 and 0 < h <= 1
            assert x - w / 2 >= -1e-7
            assert y - h / 2 >= -1e-7
            assert x + w / 2 <= 1 + 1e-7
            assert y + h / 2 <= 1 + 1e-7

        print(
            f"{split}: OK | {label_path.name} | "
            f"{len(lines)} boxes | image size={image_size}"
        )
        checked = True
        break

    assert checked, f"No positive image found in {split}"

print("All three splits passed the positive-label smoke test.")


In [ ]:

from pathlib import Path
import csv
import json
import random

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# --------------------------------------------------
# 1. Paths and dataset settings
# --------------------------------------------------
OUTPUT = Path("/kaggle/working/dfire_prepared")
DATA = OUTPUT / "data"

SPLITS = {
    "train": 14122,
    "val": 3099,
    "test": 4306,
}
CLASS_NAMES = {0: "smoke", 1: "fire"}

assert (OUTPUT / "dataset_summary.json").exists(), (
    "Run CORRECTED LABEL PREPARATION first."
)

summary = json.loads(
    (OUTPUT / "dataset_summary.json").read_text(encoding="utf-8")
)

# --------------------------------------------------
# 2. Build one correct CSV of empty-label train images
# --------------------------------------------------
image_dir = DATA / "train" / "images"
label_dir = DATA / "train" / "labels"

if not image_dir.is_dir() or not label_dir.is_dir():
    raise FileNotFoundError("Prepared train images or labels are missing.")

image_by_stem = {}
for image_path in image_dir.iterdir():
    if image_path.is_file():
        image_by_stem.setdefault(image_path.stem, image_path)

negative_rows = []
missing_images = []

for label_path in sorted(label_dir.glob("*.txt")):
    # Empty labels are candidates, not automatically confirmed hard negatives.
    if label_path.read_text(encoding="utf-8").strip():
        continue

    image_path = image_by_stem.get(label_path.stem)

    if image_path is None:
        missing_images.append(label_path.name)
        continue

    negative_rows.append({
        "image": image_path.name,
        "image_path": str(image_path),
        "image_path_relative": str(
            Path("data") / "train" / "images" / image_path.name
        ),
        "label_path_relative": str(
            Path("data") / "train" / "labels" / label_path.name
        ),
        "split": "train",
        "status": "empty_label_candidate",
        "manual_visual_review": "not individually verified",
    })

if missing_images:
    raise ValueError(
        f"{len(missing_images)} empty labels have no matching image. "
        f"Examples: {missing_images[:5]}"
    )

negative_df = pd.DataFrame(negative_rows)

negative_csv = OUTPUT / "negative_samples_train.csv"
negative_df.to_csv(negative_csv, index=False)

print("Negative-label candidates:", len(negative_df))
print("Missing corresponding images:", len(missing_images))
print("CSV:", negative_csv)

# --------------------------------------------------
# 3. Show 12 candidates for visual inspection
# --------------------------------------------------
if len(negative_df) > 0:
    sample = negative_df.sample(
        n=min(12, len(negative_df)),
        random_state=42,
    )

    fig, axes = plt.subplots(3, 4, figsize=(16, 10))
    axes = axes.flatten()

    for ax, (_, row) in zip(axes, sample.iterrows()):
        with Image.open(row["image_path"]) as im:
            ax.imshow(im.convert("RGB"))
        ax.set_title(row["image"], fontsize=8)
        ax.axis("off")

    for ax in axes[len(sample):]:
        ax.axis("off")

    plt.suptitle(
        "Empty-label train candidates — check for missed fire/smoke",
        fontsize=14,
    )
    plt.tight_layout()
    plt.show()

    # Save the exact images shown above for recording the review.
    review_csv = OUTPUT / "negative_samples_visual_review.csv"
    sample[[
        "image", "image_path_relative", "manual_visual_review"
    ]].to_csv(review_csv, index=False)

    print("Review sample CSV:", review_csv)

# --------------------------------------------------
# 4. Create a Kaggle config and a portable GitHub config
# --------------------------------------------------
kaggle_yaml = OUTPUT / "data.yaml"
kaggle_yaml.write_text(
    f"path: {DATA}\n"
    "train: train/images\n"
    "val: val/images\n"
    "test: test/images\n"
    "names:\n"
    "  0: smoke\n"
    "  1: fire\n"
    "nc: 2\n",
    encoding="utf-8",
)

# This config is for a repository containing the prepared "data/" folder.
github_yaml = OUTPUT / "data_github.yaml"
github_yaml.write_text(
    "path: ./data\n"
    "train: train/images\n"
    "val: val/images\n"
    "test: test/images\n"
    "names:\n"
    "  0: smoke\n"
    "  1: fire\n"
    "nc: 2\n",
    encoding="utf-8",
)

# --------------------------------------------------
# 5. Check the prepared dataset and build final summary
# --------------------------------------------------
split_checks = {}

for split, expected in SPLITS.items():
    split_images = DATA / split / "images"
    split_labels = DATA / split / "labels"

    image_count = sum(p.is_file() for p in split_images.iterdir())
    label_count = len(list(split_labels.glob("*.txt")))

    split_checks[split] = {
        "expected_images": expected,
        "actual_images": image_count,
        "label_files": label_count,
        "counts_match": image_count == expected == label_count,
    }

    if not split_checks[split]["counts_match"]:
        raise ValueError(
            f"{split}: image/label count check failed: "
            f"{split_checks[split]}"
        )

final_summary = {
    "dataset": "D-Fire smoke/fire detection",
    "source": (
        "https://www.kaggle.com/datasets/"
        "sayedgamal99/smoke-fire-detection-yolo"
    ),
    "classes": CLASS_NAMES,
    "split_checks": split_checks,
    "preparation_summary": summary,
    "train_empty_label_candidates": len(negative_df),
    "negative_candidate_policy": (
        "Candidates are based on empty labels; "
        "not all candidates have been manually verified."
    ),
    "original_dataset_modified": False,
    "full_model_training_performed": False,
    "final_test_evaluation_performed": False,
}

(OUTPUT / "data_lead_final_summary.json").write_text(
    json.dumps(final_summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# --------------------------------------------------
# 6. Write the Data Lead README
# --------------------------------------------------
readme = f"""# Data Preparation — Fire and Smoke Detection

## Dataset

Source: {final_summary["source"]}

Classes:
- 0: smoke
- 1: fire

Original train/validation/test splits are preserved.
The original dataset was not modified.

## Preparation

- Invalid annotation rows were removed from the prepared copy.
- Boxes crossing image boundaries were clipped to visible image bounds.
- Empty labels were retained.
- Empty-label train images are candidates for negative samples.
- Candidate images are not automatically considered verified hard negatives.

## Split verification

| Split | Expected images | Actual images | Label files | Passed |
|---|---:|---:|---:|---|
"""
for split, values in split_checks.items():
    readme += (
        f'| {split} | {values["expected_images"]} '
        f'| {values["actual_images"]} | {values["label_files"]} '
        f'| {values["counts_match"]} |\n'
    )

readme += """
## Files

- `data.yaml`: configuration for the current Kaggle environment.
- `data_github.yaml`: portable configuration for a repository
  containing the prepared `data/` directory.
- `dataset_summary.json`: annotation preparation summary.
- `data_lead_final_summary.json`: final checks and summary.
- `removed_label_rows.csv`: rejected annotation rows.
- `negative_samples_train.csv`: train empty-label candidates.
- `negative_samples_visual_review.csv`: sampled candidates shown
  for manual visual inspection.

## Limitations

Only visually inspected candidates can be described as manually reviewed.
A hard negative should be confirmed to contain a confusing background
such as clouds, fog, bright lighting, or reflective objects.
No full model training or final test evaluation is reported here.
"""

(OUTPUT / "README_DATA_LEAD.md").write_text(
    readme, encoding="utf-8"
)

# --------------------------------------------------
# 7. Final output
# --------------------------------------------------
print("\n=== DATA LEAD FINAL PREPARATION ===")
print("Split checks:")
for split, values in split_checks.items():
    print(split, values)

print("\nArtifacts:")
for name in [
    "data.yaml",
    "data_github.yaml",
    "dataset_summary.json",
    "data_lead_final_summary.json",
    "removed_label_rows.csv",
    "negative_samples_train.csv",
    "negative_samples_visual_review.csv",
    "README_DATA_LEAD.md",
]:
    path = OUTPUT / name
    if path.exists():
        print("-", path)

print("\nPrepared dataset:", DATA)
print("Original dataset unchanged.")
print("No full model training or final test evaluation performed.")


In [ ]:

# ============================================================
# DATA LEAD: FINAL PREPARATION
# Dataset audit results + negative samples + documentation
# + loader smoke tests for YOLOv8s and RT-DETR
# ============================================================

import sys
import csv
import json
import subprocess
from pathlib import Path

# -------------------- 1. PATHS --------------------

BASE = Path("/kaggle/working/dfire_prepared")
DATA = BASE / "data"
REPORT_PATH = BASE / "dataset_summary.json"

assert DATA.exists(), f"Prepared dataset not found: {DATA}"

for split in ("train", "val", "test"):
    assert (DATA / split / "images").is_dir()
    assert (DATA / split / "labels").is_dir()

print("Prepared dataset found:", DATA)

# -------------------- 2. SAVE NEGATIVE SAMPLE LIST --------------------
# Empty labels are treated as negative samples according to the
# dataset assumption and the user's visual spot-check.
# Original data is not modified.

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

negative_csv = BASE / "negative_samples_train.csv"
negative_count = 0

with negative_csv.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["image", "image_path", "label_path", "status"],
    )
    writer.writeheader()

    image_dir = DATA / "train" / "images"
    label_dir = DATA / "train" / "labels"

    for label_path in sorted(label_dir.glob("*.txt")):
        if label_path.read_text(encoding="utf-8").strip():
            continue

        matches = [
            p for p in image_dir.glob(label_path.stem + ".*")
            if p.suffix.lower() in IMAGE_EXTENSIONS
        ]

        if not matches:
            print("WARNING: image not found for", label_path.name)
            continue

        writer.writerow({
            "image": label_path.stem,
            "image_path": str(matches[0]),
            "label_path": str(label_path),
            "status": "empty_label_negative_candidate",
        })
        negative_count += 1

print("\nNegative samples saved:", negative_count)
print("File:", negative_csv)

# -------------------- 3. WRITE ULTRALYTICS DATA CONFIG --------------------

data_yaml = BASE / "data.yaml"

data_yaml.write_text(
    f'path: "{DATA}"\n'
    'train: train/images\n'
    'val: val/images\n'
    'test: test/images\n'
    'names:\n'
    '  0: smoke\n'
    '  1: fire\n'
    'nc: 2\n',
    encoding="utf-8",
)

print("\nUltralytics config saved:", data_yaml)

# -------------------- 4. SAVE AUDIT SUMMARY --------------------

summary = {
    "dataset": "D-Fire YOLO-ready dataset",
    "prepared_data_path": str(DATA),
    "ultralytics_yaml": str(data_yaml),
    "splits": {
        "train": {
            "images": 14122,
            "empty_labels": 6458,
            "boxes": 17419,
            "smoke_boxes": 7788,
            "fire_boxes": 9631,
        },
        "val": {
            "images": 3099,
            "empty_labels": 1375,
            "boxes": 3931,
            "smoke_boxes": 1755,
            "fire_boxes": 2176,
        },
        "test": {
            "images": 4306,
            "empty_labels": 2008,
            "boxes": 5181,
            "smoke_boxes": 2303,
            "fire_boxes": 2878,
        },
    },
    "total_images": 21527,
    "total_boxes": 26531,
    "total_smoke_boxes": 11846,
    "total_fire_boxes": 14685,
    "invalid_label_rows_removed": 26,
    "clipped_boxes": 371,
    "negative_train_candidates": negative_count,
    "negative_candidate_policy": (
        "Train images with empty prepared labels; "
        "not independently verified one by one."
    ),
    "original_dataset_modified": False,
    "test_split_used_for_training": False,
}

with (BASE / "data_lead_final_summary.json").open(
    "w", encoding="utf-8"
) as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

# -------------------- 5. WRITE TEAM DOCUMENTATION --------------------

readme = f"""# Fire and Smoke Detection — Data Preparation

## Dataset
D-Fire YOLO-ready dataset:
https://www.kaggle.com/datasets/sayedgamal99/smoke-fire-detection-yolo

Prepared dataset path:
`{DATA}`

Ultralytics configuration:
`{data_yaml}`

## Fixed dataset splits

| Split | Images | Empty labels | Boxes | Smoke boxes | Fire boxes |
|---|---:|---:|---:|---:|---:|
| Train | 14122 | 6458 | 17419 | 7788 | 9631 |
| Validation | 3099 | 1375 | 3931 | 1755 | 2176 |
| Test | 4306 | 2008 | 5181 | 2303 | 2878 |
| Total | 21527 | 9841 | 26531 | 11846 | 14685 |

Class IDs in YOLO labels:
- `0`: smoke
- `1`: fire

## Label preparation

- The original dataset was not modified.
- A prepared working copy was created under `dfire_prepared/data`.
- 26 invalid annotation rows were removed.
- 371 boxes crossing image boundaries were clipped to the visible image area.
- The prepared labels were checked for valid coordinates and class IDs.
- Empty label files were retained as images without annotated objects.

## Negative samples

A CSV of train images with empty prepared labels is available at:

`{negative_csv}`

Number of candidates: {negative_count}

These are treated as negative examples based on the dataset annotations.
A sample of images was visually inspected; not every image was checked
individually. The CSV does not establish that every image is a hard negative.
Hard negatives specifically contain confusing backgrounds such as clouds,
fog, bright objects, or artificial lighting.

## Model data loading

- YOLOv8s: use the Ultralytics configuration in `data.yaml`.
- RT-DETR: use the same Ultralytics configuration in `data.yaml`.
- Faster R-CNN: use the PyTorch `FireSmokeDataset` and detection collate
  function from the Data Lead notebook. Torchvision class IDs are shifted:
  smoke = 1, fire = 2, background = 0.
- Images may have different original dimensions. The Faster R-CNN dataset
  returns images in their original resolution; the torchvision detection
  model applies its own resizing pipeline.

## Split policy

- Train is used for model fitting.
- Validation is used for model selection and tuning.
- Test is reserved for final evaluation and must not be used for training
  or tuning.

## Important limitations

- A successful data audit does not prove model quality.
- The negative candidate list was generated from empty labels, not from
  a manual review of every image.
- A loader smoke test is not a substitute for full training or evaluation.
- Record mAP, precision, recall, and false-positive categories during
  the model evaluation stage.
"""

readme_path = BASE / "README_DATA_LEAD.md"
readme_path.write_text(readme, encoding="utf-8")

print("\nDocumentation saved:", readme_path)

# -------------------- 6. INSTALL ULTRALYTICS --------------------
# Required for Ultralytics' native YOLO and RT-DETR loaders.
# This does not install or train any model weights by itself.

print("\nInstalling Ultralytics and PyYAML...")
install_result = subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "ultralytics", "pyyaml"
    ],
    text=True,
    capture_output=True,
)

if install_result.returncode != 0:
    print("INSTALL FAILED:")
    print(install_result.stderr[-3000:])
else:
    print("Ultralytics and PyYAML installation command completed.")

# -------------------- 7. OPTIONAL LOADER SMOKE TESTS --------------------
# These run a tiny one-epoch training smoke test (0.1% of train data)
# to verify that Ultralytics can read the dataset and start training.
# They are NOT final model training.
#
# If pretrained weights cannot be downloaded, the test will be recorded
# as blocked instead of being reported as passed.

loader_tests = {}

if install_result.returncode == 0:
    try:
        import ultralytics
        import yaml
        from ultralytics import YOLO

        print("\nUltralytics version:", ultralytics.__version__)

        with data_yaml.open(encoding="utf-8") as f:
            cfg = yaml.safe_load(f)

        assert cfg["nc"] == 2
        assert cfg["names"][0] == "smoke"
        assert cfg["names"][1] == "fire"

        # Run a small loader/training smoke test for each architecture.
        # 0.1% of 14,122 train images is about 14 images.
        test_models = {
            "YOLOv8s": "yolov8s.pt",
            "RT-DETR": "rtdetr-l.pt",
        }

        for model_name, weights in test_models.items():
            print(f"\nStarting {model_name} smoke test...")
            try:
                model = YOLO(weights)

                model.train(
                    data=str(data_yaml),
                    epochs=1,
                    fraction=0.001,
                    imgsz=640,
                    batch=2,
                    workers=2,
                    device=0 if __import__("torch").cuda.is_available() else "cpu",
                    val=False,
                    save=False,
                    plots=False,
                    verbose=False,
                    project=str(BASE / "loader_smoke_tests"),
                    name=model_name.lower().replace("-", "_"),
                    exist_ok=True,
                )

                loader_tests[model_name] = "PASSED: tiny smoke test completed"
                print(model_name, "smoke test completed.")

            except Exception as e:
                loader_tests[model_name] = (
                    f"BLOCKED/FAILED: {type(e).__name__}: {e}"
                )
                print(model_name, "smoke test did not complete:", repr(e))

    except Exception as e:
        loader_tests["Ultralytics setup"] = (
            f"BLOCKED/FAILED: {type(e).__name__}: {e}"
        )
        print("\nUltralytics setup/check failed:", repr(e))
else:
    loader_tests["Ultralytics setup"] = (
        "BLOCKED: package installation failed"
    )

# -------------------- 7B. FASTER R-CNN DATALOADER TEST --------------------
# This verifies data loading and target formatting only.
# It does NOT train Faster R-CNN.

print("\n" + "=" * 55)
print("FASTER R-CNN DATALOADER SMOKE TEST")
print("=" * 55)

try:
    import math
    import numpy as np
    import torch
    from PIL import Image
    from torch.utils.data import Dataset, DataLoader

    class FireSmokeDataset(Dataset):
        def __init__(self, data_dir, split="train"):
            self.image_dir = Path(data_dir) / split / "images"
            self.label_dir = Path(data_dir) / split / "labels"

            extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
            self.images = sorted(
                p for p in self.image_dir.iterdir()
                if p.is_file() and p.suffix.lower() in extensions
            )

            if not self.images:
                raise RuntimeError(
                    f"No images found in {self.image_dir}"
                )

            # Every image must have a corresponding YOLO label file.
            missing = [
                p.name for p in self.images
                if not (self.label_dir / f"{p.stem}.txt").is_file()
            ]
            if missing:
                raise RuntimeError(
                    f"{len(missing)} images have no label file. "
                    f"Examples: {missing[:5]}"
                )

        def __len__(self):
            return len(self.images)

        def __getitem__(self, index):
            image_path = self.images[index]
            label_path = self.label_dir / f"{image_path.stem}.txt"

            # Read image and convert it to an RGB float tensor [C, H, W].
            with Image.open(image_path) as img:
                img = img.convert("RGB")
                image_array = np.array(img, copy=True)
                width, height = img.size

            image = torch.from_numpy(image_array).permute(2, 0, 1)
            image = image.float() / 255.0

            boxes = []
            labels = []

            text = label_path.read_text(encoding="utf-8").strip()

            if text:
                for line_number, line in enumerate(text.splitlines(), 1):
                    parts = line.split()

                    if len(parts) != 5:
                        raise ValueError(
                            f"Malformed label: {label_path.name}, "
                            f"line {line_number}"
                        )

                    class_id = int(parts[0])
                    cx, cy, w, h = map(float, parts[1:])

                    if class_id not in (0, 1):
                        raise ValueError(
                            f"Unexpected class ID {class_id} "
                            f"in {label_path.name}"
                        )

                    if not all(
                        math.isfinite(v) for v in (cx, cy, w, h)
                    ):
                        raise ValueError(
                            f"Non-finite coordinates in {label_path.name}"
                        )

                    if w <= 0 or h <= 0:
                        raise ValueError(
                            f"Non-positive box size in {label_path.name}"
                        )

                    # YOLO normalized xywh -> absolute xyxy pixel coordinates.
                    x1 = (cx - w / 2) * width
                    y1 = (cy - h / 2) * height
                    x2 = (cx + w / 2) * width
                    y2 = (cy + h / 2) * height

                    # Clamp small floating-point boundary deviations.
                    x1 = max(0.0, min(float(width), x1))
                    y1 = max(0.0, min(float(height), y1))
                    x2 = max(0.0, min(float(width), x2))
                    y2 = max(0.0, min(float(height), y2))

                    if x2 <= x1 or y2 <= y1:
                        raise ValueError(
                            f"Invalid box after conversion in "
                            f"{label_path.name}, line {line_number}"
                        )

                    boxes.append([x1, y1, x2, y2])

                    # Torchvision: 0 is background; object IDs start at 1.
                    # Original YOLO: 0=smoke, 1=fire.
                    labels.append(class_id + 1)

            boxes_tensor = torch.tensor(
                boxes, dtype=torch.float32
            ).reshape(-1, 4)

            labels_tensor = torch.tensor(
                labels, dtype=torch.int64
            )

            target = {
                "boxes": boxes_tensor,
                "labels": labels_tensor,
                "image_id": torch.tensor([index]),
                "area": (
                    (boxes_tensor[:, 2] - boxes_tensor[:, 0])
                    * (boxes_tensor[:, 3] - boxes_tensor[:, 1])
                ),
                "iscrowd": torch.zeros(
                    len(boxes_tensor), dtype=torch.int64
                ),
            }

            return image, target

    def detection_collate_fn(batch):
        return tuple(zip(*batch))

    faster_rcnn_dataset = FireSmokeDataset(DATA, split="train")

    faster_rcnn_loader = DataLoader(
        faster_rcnn_dataset,
        batch_size=4,
        shuffle=False,
        num_workers=0,
        collate_fn=detection_collate_fn,
    )

    images, targets = next(iter(faster_rcnn_loader))

    # Basic batch and tensor checks.
    assert len(images) == 4, "Expected a batch of four images."

    for image, target in zip(images, targets):
        assert image.ndim == 3 and image.shape[0] == 3
        assert image.dtype == torch.float32
        assert torch.isfinite(image).all()
        assert image.min() >= 0 and image.max() <= 1

        boxes = target["boxes"]
        labels = target["labels"]

        assert boxes.ndim == 2 and boxes.shape[1] == 4
        assert labels.ndim == 1
        assert len(boxes) == len(labels)
        assert torch.isfinite(boxes).all()

        # Valid object class IDs for Torchvision detection.
        assert torch.all((labels == 1) | (labels == 2))

        height, width = image.shape[1:]

        if len(boxes) > 0:
            assert torch.all(boxes[:, 0] >= 0)
            assert torch.all(boxes[:, 1] >= 0)
            assert torch.all(boxes[:, 2] <= width)
            assert torch.all(boxes[:, 3] <= height)
            assert torch.all(boxes[:, 2] > boxes[:, 0])
            assert torch.all(boxes[:, 3] > boxes[:, 1])

        assert len(target["area"]) == len(boxes)
        assert len(target["iscrowd"]) == len(boxes)

    loader_tests["Faster R-CNN DataLoader"] = (
        "PASSED: dataset and batch checks completed"
    )

    print("Dataset images:", len(faster_rcnn_dataset))
    print("Batch size:", len(images))
    print("First image shape:", tuple(images[0].shape))
    print("First image boxes:", len(targets[0]["boxes"]))
    print("First image labels:", targets[0]["labels"].tolist())
    print("Faster R-CNN DataLoader test: PASSED")

except Exception as e:
    loader_tests["Faster R-CNN DataLoader"] = (
        f"BLOCKED/FAILED: {type(e).__name__}: {e}"
    )
    print("Faster R-CNN DataLoader test: FAILED")
    print(repr(e))

# -------------------- 8. SAVE FINAL STATUS --------------------

summary["ultralytics_loader_smoke_tests"] = loader_tests
summary["documentation"] = str(readme_path)
summary["negative_samples_csv"] = str(negative_csv)

with (BASE / "data_lead_final_summary.json").open(
    "w", encoding="utf-8"
) as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 55)
print("DATA LEAD PREPARATION SUMMARY")
print("=" * 55)
print("Prepared dataset:", DATA)
print("YOLO / RT-DETR YAML:", data_yaml)
print("Negative samples CSV:", negative_csv)
print("README:", readme_path)
print("Final JSON summary:", BASE / "data_lead_final_summary.json")
print("\nUltralytics loader smoke-test results:")
for name, result in loader_tests.items():
    print(f"- {name}: {result}")

print("\nOriginal dataset unchanged.")
print("No full model training or final test evaluation was performed.")


In [ ]:
from pathlib import Path
import zipfile

ROOT = Path("/kaggle/working/dfire_prepared")
ARCHIVE = Path("/kaggle/working/data_lead_artifacts.zip")

# Берём только те файлы, которые действительно существуют.
files_to_include = [
    "README_DATA_LEAD.md",
    "data.yaml",
    "dataset_summary.json",
    "removed_label_rows.csv",
    "negative_samples_train.csv",
    "data_lead_final_summary.json",
]

with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as z:
    for filename in files_to_include:
        path = ROOT / filename
        if path.is_file():
            z.write(path, arcname=filename)
            print("Added:", filename)
        else:
            print("NOT FOUND:", filename)

print("\nArchive:", ARCHIVE)
print("Size:", round(ARCHIVE.stat().st_size / 1024, 1), "KB")